In [ ]:
import json
from pathlib import Path
from collections import Counter, defaultdict

# COCO 어노테이션 JSON 경로 (train / valid / test 중 골라서 바꾸면 됨)
COCO_JSON = Path("mechanical-parts-coco/train/_annotations.coco.json")  # 리포 루트에서 실행
IMG_DIR = COCO_JSON.parent  # 이미지들은 JSON과 같은 폴더에 있음

data = json.loads(COCO_JSON.read_text(encoding="utf-8"))

# COCO JSON의 최상위 구조(키)와 개수 요약
print("최상위 키:", list(data.keys()))
print("이미지 수     :", len(data["images"]))
print("어노테이션 수 :", len(data["annotations"]), "(=바운딩 박스 총 개수)")
print("카테고리 수   :", len(data["categories"]))

In [ ]:
# 1) 카테고리(클래스) 목록과 클래스별 박스 개수
id2name = {c["id"]: c["name"] for c in data["categories"]}
print("카테고리:")
for c in data["categories"]:
    print(f"  id={c['id']:>2}  name={c['name']:<12}  supercategory={c.get('supercategory')}")

# 어노테이션을 클래스별로 세어 본다 (라벨이 실제로 붙어 있는지 확인)
cnt = Counter(a["category_id"] for a in data["annotations"])
print("\n클래스별 박스 개수:")
for cid, n in sorted(cnt.items()):
    print(f"  {id2name[cid]:<12} : {n}개")

In [ ]:
# 2) 특정 이미지 하나의 라벨을 사람이 읽기 좋게 출력
#    image_id -> 그 이미지에 달린 어노테이션들 매핑
anns_by_img = defaultdict(list)
for a in data["annotations"]:
    anns_by_img[a["image_id"]].append(a)

# 첫 번째 이미지를 예시로 (원하는 파일명으로 바꿔도 됨)
img = data["images"][0]
print(f"파일명 : {img['file_name']}")
print(f"크기   : {img['width']} x {img['height']} (WxH)")
print(f"박스 수: {len(anns_by_img[img['id']])}\n")

# COCO bbox 포맷 = [x_min, y_min, width, height] (픽셀 단위)
print(f"{'클래스':<10}{'x_min':>8}{'y_min':>8}{'w':>8}{'h':>8}")
for a in anns_by_img[img["id"]]:
    x, y, w, h = a["bbox"]
    print(f"{id2name[a['category_id']]:<10}{x:>8.1f}{y:>8.1f}{w:>8.1f}{h:>8.1f}")

In [ ]:
# 3) 이미지 위에 바운딩 박스를 그려서 눈으로 확인 (matplotlib, pillow 필요)
#    pip install matplotlib pillow
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

img = data["images"][0]              # 원하는 인덱스로 변경 가능
im = Image.open(IMG_DIR / img["file_name"]).convert("RGB")

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(im)
for a in anns_by_img[img["id"]]:
    x, y, w, h = a["bbox"]
    ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor="red", linewidth=2))
    ax.text(x, y - 4, id2name[a["category_id"]], color="white",
            fontsize=10, bbox=dict(facecolor="red", pad=1, edgecolor="none"))
ax.set_title(f"{img['file_name']}  (박스 {len(anns_by_img[img['id']])}개)")
ax.axis("off")
plt.show()

In [ ]:
# 4) (선택) JSON 원문을 그대로 들여다보기 - 앞부분만 예쁘게 출력
print("== categories 원문 ==")
print(json.dumps(data["categories"], ensure_ascii=False, indent=2))

print("\n== images[0] 원문 ==")
print(json.dumps(data["images"][0], ensure_ascii=False, indent=2))

print("\n== annotations[0] 원문 ==")
print(json.dumps(data["annotations"][0], ensure_ascii=False, indent=2))